# Tarea 2 — Entrenamiento encoder (binario)\n\nEste notebook entrena modelos tipo encoder para **detección binaria de contribuciones** (`is_contribution`), usando:\n- **Train (silver):** `task2_contributions_silver_binary.parquet`\n- **Eval (gold):** `task2_gold.parquet`\n\nTambién **loguea a MLflow en Drive** (file store), igual que en el notebook de Tarea 1.

In [2]:
!pip -q install mlflow evaluate datasets transformers safetensors accelerate scikit-learn pandas pyarrow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 14.3 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os, mlflow
TRACKING_DIR = "/content/drive/MyDrive/class_runs"
os.makedirs(TRACKING_DIR, exist_ok=True)
mlflow.set_tracking_uri(f"file:{TRACKING_DIR}")
mlflow.set_experiment("task2_contributions_binary")

print("MLflow tracking:", mlflow.get_tracking_uri())
print("Experiment:", "task2_contributions_binary")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


MLflow tracking: file:/content/drive/MyDrive/class_runs
Experiment: task2_contributions_binary


## 1) Cargar parquets (silver + gold)\n\nSube estos archivos a Colab (o Drive) y ajusta rutas.

In [4]:
import pandas as pd

# Ajusta rutas (sube a Colab o monta desde Drive)
silver_path = "/content/task2_contributions_silver_binary.parquet"
gold_path   = "/content/task2_gold.parquet"

silver = pd.read_parquet(silver_path)
gold   = pd.read_parquet(gold_path)


print("silver columns:", silver.columns.tolist())
print("gold columns:", gold.columns.tolist())
print("silver rows:", len(silver))
print("gold rows:", len(gold))

silver columns: ['fragment_id', 'source_chunk_id', 'doc_id', 'source_path', 'heading', 'n_words', 'text', 'is_contribution']
gold columns: ['fragment_id', 'source_chunk_id', 'doc_id', 'source_path', 'heading', 'n_words', 'text', 'is_contribution']
silver rows: 2000
gold rows: 191


In [5]:
def as_int_label(v):
    # Accept bool/0/1/"true"/"false"
    if v is None:
        return 0
    if isinstance(v, bool):
        return 1 if v else 0
    if isinstance(v, (int, float)):
        return 1 if int(v) == 1 else 0
    s = str(v).strip().lower()
    if s in {"1","true","t","yes","y","si","sí","pos","positive"}:
        return 1
    return 0

for df in (silver, gold):
    if "text" not in df.columns or "is_contribution" not in df.columns:
        raise ValueError("Expected columns: text, is_contribution")
    df["text"] = df["text"].astype(str)
    df["labels"] = df["is_contribution"].apply(as_int_label).astype(int)

print("silver positives:", int(silver["labels"].sum()), "negatives:", int(len(silver)-silver["labels"].sum()))
print("gold positives:", int(gold["labels"].sum()), "negatives:", int(len(gold)-gold["labels"].sum()))
silver[["labels"]].value_counts()

silver positives: 1000 negatives: 1000
gold positives: 52 negatives: 139


,count
labels,
0,1000
1,1000


## 2) Trainer (Transformers) + métricas\n\nEntrena un clasificador binario (`NEG` vs `POS`) y loguea todo a MLflow (métricas + carpeta HF exportada como artefacto).

In [7]:
import numpy as np
import evaluate
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

metric_f1 = evaluate.load("f1")
metric_acc = evaluate.load("accuracy")
metric_prec = evaluate.load("precision")
metric_rec = evaluate.load("recall")

LABELS = ["NEG","POS"]
label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for l,i in label2id.items()}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = np.exp(logits) / np.exp(logits).sum(axis=-1, keepdims=True)
    preds = np.argmax(probs, axis=-1)
    out = {
        "accuracy": metric_acc.compute(predictions=preds, references=labels)["accuracy"],
        "f1": metric_f1.compute(predictions=preds, references=labels, average="binary")["f1"],
        "precision": metric_prec.compute(predictions=preds, references=labels, average="binary")["precision"],
        "recall": metric_rec.compute(predictions=preds, references=labels, average="binary")["recall"],
    }
    return {k: float(v) for k,v in out.items()}

def train_transformer(model_name_or_path, run_name, max_length=256, lr=2e-5, epochs=2, batch_size=16):
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    def tok(batch):
        return tokenizer(batch["text"], truncation=True, max_length=max_length)

    ds_train = Dataset.from_pandas(silver[["text","labels"]])
    ds_eval  = Dataset.from_pandas(gold[["text","labels"]])
    ds_train = ds_train.map(tok, batched=True)
    ds_eval  = ds_eval.map(tok, batched=True)
    ds_train = ds_train.remove_columns([c for c in ds_train.column_names if c not in ["input_ids","attention_mask","labels"]])
    ds_eval  = ds_eval.remove_columns([c for c in ds_eval.column_names  if c not in ["input_ids","attention_mask","labels"]])

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name_or_path,
        num_labels=2,
        label2id=label2id,
        id2label=id2label,
        ignore_mismatched_sizes=True,  # útil si inicializas desde un modelo de Task1 (8 labels)
    )

    export_dir = f"./export_{run_name}"
    args = TrainingArguments(
        output_dir=f"./out_{run_name}",
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        save_strategy="no",
        logging_steps=50,
        report_to=[],
    )

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            "task": "task2_binary",
            "model_name": model_name_or_path,
            "max_length": max_length,
            "lr": lr,
            "epochs": epochs,
            "batch_size": batch_size,
        })

        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=ds_train,
            eval_dataset=ds_eval,
            data_collator=collator,
            processing_class=tokenizer,
            compute_metrics=compute_metrics,
        )

        trainer.train()
        trainer.save_model(export_dir)
        tokenizer.save_pretrained(export_dir)

        metrics = trainer.evaluate()
        for k, v in metrics.items():
            if isinstance(v, (int, float)):
                mlflow.log_metric(k, float(v))

        # log carpeta HF para luego descargar/subir (hf_model)
        mlflow.log_artifacts(local_dir=export_dir, artifact_path="hf_model")

        print("Run ID:", mlflow.active_run().info.run_id)
        print("Artifact URI:", mlflow.get_artifact_uri("hf_model"))

    return export_dir


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## 3) Entrenar modelos base (RoBERTa / BERT / SciBETO)\n\nPuedes cambiar hiperparámetros según tu presupuesto de GPU.

In [7]:
# Modelo A (RoBERTa BNE)
train_transformer(
    model_name_or_path="Buzzeitor/roberta-base-bne",
    run_name="task2_roberta_bne",
    max_length=256,
    lr=2e-5,
    epochs=2,
    batch_size=16,
)

config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Step,Training Loss
50,0.686698
100,0.674970
150,0.656516
200,0.598032
250,0.614009


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Run ID: cdde23c7aeb647559d9383c9f7702bdd
Artifact URI: file:///content/drive/MyDrive/class_runs/906214974828521023/cdde23c7aeb647559d9383c9f7702bdd/artifacts/hf_model


'./export_task2_roberta_bne'

In [8]:
# Modelo B (BERT Spanish WWM)
train_transformer(
    model_name_or_path="dccuchile/bert-base-spanish-wwm-cased",
    run_name="task2_bert_spanish_wwm",
    max_length=256,
    lr=2e-5,
    epochs=2,
    batch_size=16,
)

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Step,Training Loss
50,0.681221
100,0.681161
150,0.625969
200,0.569709
250,0.578177


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Run ID: 32d1fb939f7f4df087a7bdfe095f9bde
Artifact URI: file:///content/drive/MyDrive/class_runs/906214974828521023/32d1fb939f7f4df087a7bdfe095f9bde/artifacts/hf_model


'./export_task2_bert_spanish_wwm'

In [9]:
# Modelo C (SciBETO Large)
# Nota: usa GPU en Colab para este.
train_transformer(
    model_name_or_path="Flaglab/SciBETO-large",
    run_name="task2_scibeto_large",
    max_length=256,
    lr=2e-5,
    epochs=2,
    batch_size=8,
)

config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/378 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

RobertaForSequenceClassification LOAD REPORT from: Flaglab/SciBETO-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
50,0.692228
100,0.679502
150,0.708499
200,0.637817
250,0.614562
300,0.473153
350,0.412980
400,0.417083
450,0.408532
500,0.455128


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Run ID: d16e11ac666748cfa5173e8280d00800
Artifact URI: file:///content/drive/MyDrive/class_runs/906214974828521023/d16e11ac666748cfa5173e8280d00800/artifacts/hf_model


'./export_task2_scibeto_large'

# Task
Re-train the following models (`Buzzeitor/roberta-base-bne`, `dccuchile/bert-base-spanish-wwm-cased`, and `Flaglab/SciBETO-large`) using 3 and 4 epochs each. Ensure all training runs are logged to MLflow, then summarize the findings across all models and epoch configurations.

## Entrenar RoBERTa BNE con diferentes épocas

### Subtask:
Re-entrenar el modelo `Buzzeitor/roberta-base-bne` con diferentes números de épocas (e.g., 3 y 4), manteniendo los otros hiperparámetros constantes. Cada entrenamiento se registrará en MLflow.


**Reasoning**:
The user wants to re-train the `Buzzeitor/roberta-base-bne` model with 3 epochs, keeping other hyperparameters constant. I will use the existing `train_transformer` function for this.



In [11]:
train_transformer(
    model_name_or_path="Buzzeitor/roberta-base-bne",
    run_name="task2_roberta_bne_epochs_3",
    max_length=256,
    lr=2e-5,
    epochs=3,
    batch_size=16,
)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/191 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: Buzzeitor/roberta-base-bne
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
50,0.678921
100,0.677857
150,0.660752
200,0.607919
250,0.619304
300,0.485855
350,0.474651


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Run ID: eb9175bd3e5c45c6a4e4ef24cb94358c
Artifact URI: file:///content/drive/MyDrive/class_runs/906214974828521023/eb9175bd3e5c45c6a4e4ef24cb94358c/artifacts/hf_model


'./export_task2_roberta_bne_epochs_3'

**Reasoning**:
The user wants to re-train the `Buzzeitor/roberta-base-bne` model with 4 epochs, keeping other hyperparameters constant. I will use the existing `train_transformer` function for this, specifying 4 epochs and a new run name.



In [12]:
train_transformer(
    model_name_or_path="Buzzeitor/roberta-base-bne",
    run_name="task2_roberta_bne_epochs_4",
    max_length=256,
    lr=2e-5,
    epochs=4,
    batch_size=16,
)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/191 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: Buzzeitor/roberta-base-bne
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
50,0.682428
100,0.677630
150,0.654116
200,0.556235
250,0.574677
300,0.350506
350,0.315982
400,0.213681
450,0.116375
500,0.079160


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Run ID: 1d71863f84ab4087a06c57156534d050
Artifact URI: file:///content/drive/MyDrive/class_runs/906214974828521023/1d71863f84ab4087a06c57156534d050/artifacts/hf_model


'./export_task2_roberta_bne_epochs_4'

In [ ]:
train_transformer(
    model_name_or_path="Flaglab/SciBETO-large",
    run_name="task2_scibeto_large_4_epochs",
    max_length=256,
    lr=2e-5,
    epochs=4,
    batch_size=8,
)

config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/378 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/191 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: Flaglab/SciBETO-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' a

Step,Training Loss
